# Lightweight Cue-On ITC: All ROIs

Compute Tensorpac inter-trial coherence (ITC) for `cue_on` epochs in all available `HPC`, `AMY`, `OFC`, `dmPFC`, `dlPFC`, and `ACC` channels.

Workflow:
- load MemoryBandit subjects from `MemoryBanditsDatabase.xlsx`
- load cue-on epochs and rereferenced electrode tables per subject
- merge cue-on trials with preprocessed behavior tables
- compute univariate Tensorpac `ITC(data, sf, f_pha=..., edges=..., cycle=...)` one channel at a time
- save per-channel `.npz` arrays plus metadata for channel-level pooling
- render ROI-level all-trial and RPE-category aggregate heatmaps

Tensorpac reference: https://etiennecmb.github.io/tensorpac/auto_examples/misc/plot_itc.html

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import sys
import json
import math
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# Force local Tensorpac checkout, matching the existing PAC notebook pattern.
TP_LOCAL_ROOT = '/Users/salmanqasim/Documents/GitRepos/tensorpac'
if TP_LOCAL_ROOT in sys.path:
    sys.path.remove(TP_LOCAL_ROOT)
sys.path.insert(0, TP_LOCAL_ROOT)

# Tensorpac imports Numba PAC kernels at package import time. In some local
# notebook/shell environments, Numba cannot create cache locators for this
# checkout, so disable only Numba's on-disk function cache before import.
try:
    import numba as _numba
    _numba_jit = _numba.jit

    def _jit_no_cache(*args, **kwargs):
        kwargs['cache'] = False
        return _numba_jit(*args, **kwargs)

    _numba.jit = _jit_no_cache
except Exception as exc:
    print('Numba cache guard skipped:', exc)

import tensorpac
from tensorpac.utils import ITC

from mv_granger_utils import (
    load_behavior_tables,
    preprocess_learn_df,
    preprocess_combined_df,
    get_roi_channels,
)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 150
pd.set_option('display.max_colwidth', 160)

print('tensorpac.__version__ =', getattr(tensorpac, '__version__', 'unknown'))
print('tensorpac.__file__ =', tensorpac.__file__)
assert '/Users/salmanqasim/Documents/GitRepos/tensorpac/tensorpac/' in tensorpac.__file__.replace('\\', '/'), (
    'Notebook is not using the local Tensorpac checkout. Restart kernel and run this cell first.'
)

In [ ]:
# ---- Config ----
base_dir = Path('/Volumes/T7 Shield')
load_dir = base_dir / 'work/qasims01/MemoryBanditData/EMU'
ephys_root = base_dir / 'projects/guLab/Salman/EphysAnalyses'
demo_file = base_dir / 'projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'

event = 'cue_on'
day = 'Day1'
condition = 'Day1'
rois = ['HPC', 'AMY', 'OFC', 'dmPFC', 'dlPFC', 'ACC']

# Tensorpac ITC parameters. f_pha tuple follows Tensorpac dynamic frequency-vector syntax.
f_pha = (2, 30, 1, 0.5)
dcomplex = 'wavelet'
cycle = 3
buf_ms = 1000
min_trials = 5
n_jobs = 1  # Tensorpac ITC jobs within each channel. Keep low when parallelizing subjects.
subject_n_jobs = 4  # Joblib workers over subjects; use 1 for serial debugging.
subject_backend = 'loky'  # Process-based backend keeps MNE/Tensorpac work isolated.

# Execution controls.
run_smoke = True
run_batch = True  # Set True after the smoke run passes.
smoke_subject = None  # None = first subject with at least one requested ROI cue_on channel
max_subjects = None   # Set to an int for a limited batch.
overwrite_itc = False

analysis_root = Path('/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/analysis_outputs/cue_on_itc_all_rois')
itc_root = analysis_root / 'itc_npz'
figure_root = analysis_root / 'figures'
analysis_root.mkdir(parents=True, exist_ok=True)
itc_root.mkdir(parents=True, exist_ok=True)
figure_root.mkdir(parents=True, exist_ok=True)

metadata_csv = analysis_root / 'cue_on_itc_channel_metadata.csv'
missing_csv = analysis_root / 'cue_on_itc_missing_subject_channel_report.csv'
shape_check_csv = analysis_root / 'cue_on_itc_shape_checks.csv'

print('analysis_root:', analysis_root)
print('event:', event, '| rois:', rois)
print('f_pha:', f_pha, '| dcomplex:', dcomplex, '| cycle:', cycle, '| min_trials:', min_trials)

In [ ]:
# ---- Subjects and behavior ----
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())
if max_subjects is not None:
    subjects = subjects[:int(max_subjects)]

learn_df_raw, combined_df_raw = load_behavior_tables(str(load_dir))
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print('Subjects:', len(subjects))
print('learn_df:', learn_df.shape)
print('combined_df:', combined_df.shape)
print('combined required columns present:', {'participant', 'condition', 'rpe_cat'}.issubset(combined_df.columns))
print('combined trial columns:', [c for c in ['trials', 'trials_mem', 'trials_gamble'] if c in combined_df.columns])

In [ ]:
# ---- Helpers ----
def sanitize_token(value):
    s = str(value).strip()
    keep = []
    for ch in s:
        if ch.isalnum() or ch in ['-', '_', '.']:
            keep.append(ch)
        else:
            keep.append('-')
    return ''.join(keep).strip('-') or 'NA'


def canon_rpe_cat(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    mapping = {
        'negative': 'neg',
        'neg': 'neg',
        'neutral': 'neutral',
        'positive': 'pos',
        'pos': 'pos',
    }
    return mapping.get(s, np.nan)


def cue_behavior_for_subject(subject):
    cdf = combined_df[combined_df['participant'].astype(str) == str(subject)].copy()
    if 'condition' in cdf.columns:
        cdf = cdf[cdf['condition'].astype(str) == condition].copy()
    if 'trials' not in cdf.columns:
        if 'trials_mem' in cdf.columns:
            cdf = cdf.rename(columns={'trials_mem': 'trials'})
        elif 'trials_gamble' in cdf.columns:
            cdf = cdf.rename(columns={'trials_gamble': 'trials'})
    if 'trials' in cdf.columns:
        cdf['trials'] = pd.to_numeric(cdf['trials'], errors='coerce')
        cdf = cdf.dropna(subset=['trials']).copy()
        cdf['trials'] = cdf['trials'].astype(int)
    if 'rpe_cat' in cdf.columns:
        cdf['rpe_cat'] = cdf['rpe_cat'].map(canon_rpe_cat)
    return cdf


def load_subject_inputs(subject):
    epochs_file = ephys_root / subject / 'neural' / day / f'{event}-epo.fif'
    elec_file = ephys_root / subject / 'Day1_reref_elec_df'
    if not epochs_file.exists():
        return None, None, {'participant': subject, 'item_type': 'epochs', 'detail': str(epochs_file)}
    if not elec_file.exists():
        return None, None, {'participant': subject, 'item_type': 'electrodes', 'detail': str(elec_file)}
    epochs = mne.read_epochs(str(epochs_file), preload=True, verbose=False)
    elec_df = pd.read_csv(elec_file)
    return epochs, elec_df, None


def trial_table_for_epochs(subject, n_trials):
    base = pd.DataFrame({'participant': str(subject), 'trials': np.arange(1, int(n_trials) + 1, dtype=int)})
    cdf = cue_behavior_for_subject(subject)
    if 'trials' not in cdf.columns or len(cdf) == 0:
        base['rpe_cat'] = np.nan
        return base
    cols = [c for c in cdf.columns if c not in base.columns or c in ['participant', 'trials']]
    merged = base.merge(cdf[cols].drop_duplicates(subset=['participant', 'trials']), on=['participant', 'trials'], how='left')
    if 'rpe_cat' not in merged.columns:
        merged['rpe_cat'] = np.nan
    else:
        merged['rpe_cat'] = merged['rpe_cat'].map(canon_rpe_cat)
    return merged


def trimmed_times(times, edges):
    times = np.asarray(times, dtype=float)
    edges = int(edges) if edges is not None else 0
    if edges <= 0:
        return times
    if 2 * edges >= len(times):
        raise ValueError(f'edges={edges} removes all time samples from n_times={len(times)}')
    return times[edges:-edges]


def compute_channel_itc(data_2d, sfreq, edges):
    data_2d = np.asarray(data_2d, dtype=np.float64)
    if data_2d.ndim != 2:
        raise ValueError(f'Expected data shaped (n_trials, n_times), got {data_2d.shape}')

    # Tensorpac 0.6.5 uses identity checks like `dcomplex is 'wavelet'` in
    # spectral.py. Loky pickling can de-intern strings in worker processes,
    # which makes that branch fail on Python 3.13. Re-intern locally per job.
    dcomplex_arg = sys.intern(str(dcomplex))

    itc_obj = ITC(
        data_2d,
        float(sfreq),
        f_pha=f_pha,
        dcomplex=dcomplex_arg,
        edges=int(edges),
        cycle=cycle,
        n_jobs=n_jobs,
        verbose=False,
    )
    itc_map = np.asarray(itc_obj.itc, dtype=np.float32)
    freqs = np.asarray(itc_obj.xvec, dtype=np.float32)
    if itc_map.ndim != 2:
        itc_map = np.atleast_2d(itc_map)
    return itc_map, freqs


def itc_output_path(subject, roi, channel, condition_type, condition_label):
    fname = '_'.join([
        sanitize_token(subject),
        event,
        sanitize_token(roi),
        sanitize_token(channel),
        sanitize_token(condition_type),
        sanitize_token(condition_label),
    ]) + '.npz'
    return itc_root / fname


def save_itc_npz(path, itc_map, freqs, times, times_full, subject, roi, channel, condition_type, condition_label, n_trials, sfreq, edges):
    np.savez_compressed(
        path,
        itc=np.asarray(itc_map, dtype=np.float32),
        freqs=np.asarray(freqs, dtype=np.float32),
        times=np.asarray(times, dtype=np.float32),
        times_full=np.asarray(times_full, dtype=np.float32),
        f_pha=np.asarray(f_pha, dtype=np.float32),
        subject=str(subject),
        roi=str(roi),
        channel=str(channel),
        condition_type=str(condition_type),
        condition_label=str(condition_label),
        event=event,
        n_trials=int(n_trials),
        sfreq=float(sfreq),
        edges=int(edges),
        dcomplex=dcomplex,
        cycle=int(cycle),
    )


def metadata_row(subject, roi, channel, condition_type, condition_label, n_trials, sfreq, itc_file, itc_shape, edges, status='computed', note=''):
    return {
        'participant': str(subject),
        'event': event,
        'roi': str(roi),
        'channel': str(channel),
        'condition_type': str(condition_type),
        'condition_label': str(condition_label),
        'n_trials': int(n_trials) if pd.notna(n_trials) else 0,
        'sfreq': float(sfreq) if pd.notna(sfreq) else np.nan,
        'itc_file': str(itc_file) if itc_file else '',
        'itc_shape': str(tuple(itc_shape)) if itc_shape else '',
        'edges': int(edges) if pd.notna(edges) else np.nan,
        'status': status,
        'note': note,
    }


def compute_subject_itc(subject, smoke=False):
    rows = []
    report_rows = []
    epochs, elec_df, missing = load_subject_inputs(subject)
    if missing is not None:
        report_rows.append(missing)
        return rows, report_rows

    sfreq = float(epochs.info['sfreq'])
    edges = int(1000 * sfreq / 1000)
    times_full = np.asarray(epochs.times, dtype=float)
    times = trimmed_times(times_full, edges)
    trial_df = trial_table_for_epochs(subject, len(epochs))

    for roi in rois:
        roi_info = get_roi_channels(elec_df, epochs, roi)
        channels = list(roi_info['channels'])
        if len(channels) == 0:
            report_rows.append({'participant': subject, 'item_type': 'roi_channels', 'roi': roi, 'detail': 'no valid epoch channels'})
            continue

        for channel in channels:
            ch_data = epochs.copy().pick([channel]).get_data()[:, 0, :]
            conditions = [('all_trials', 'all', np.ones(ch_data.shape[0], dtype=bool))]
            if 'rpe_cat' in trial_df.columns:
                for label in ['neg', 'neutral', 'pos']:
                    mask = trial_df['rpe_cat'].to_numpy(dtype=object) == label
                    conditions.append(('rpe_cat', label, mask))
            else:
                report_rows.append({'participant': subject, 'item_type': 'behavior', 'roi': roi, 'channel': channel, 'detail': 'rpe_cat missing; all_trials only'})

            for condition_type, condition_label, mask in conditions:
                idx = np.where(mask)[0]
                n_cond = int(len(idx))
                if n_cond < min_trials:
                    report_rows.append({
                        'participant': subject,
                        'item_type': 'condition_skip',
                        'roi': roi,
                        'channel': channel,
                        'condition_type': condition_type,
                        'condition_label': condition_label,
                        'n_trials': n_cond,
                        'detail': f'fewer than min_trials={min_trials}',
                    })
                    continue

                out_file = itc_output_path(subject, roi, channel, condition_type, condition_label)
                if out_file.exists() and not overwrite_itc:
                    with np.load(out_file, allow_pickle=True) as z:
                        itc_shape = tuple(z['itc'].shape)
                    rows.append(metadata_row(subject, roi, channel, condition_type, condition_label, n_cond, sfreq, out_file, itc_shape, edges, status='cached'))
                else:
                    itc_map, freqs = compute_channel_itc(ch_data[idx, :], sfreq, edges)
                    if itc_map.shape[-1] != len(times):
                        raise ValueError(f'ITC/time mismatch for {subject} {channel}: {itc_map.shape} vs len(times)={len(times)}')
                    save_itc_npz(out_file, itc_map, freqs, times, times_full, subject, roi, channel, condition_type, condition_label, n_cond, sfreq, edges)
                    rows.append(metadata_row(subject, roi, channel, condition_type, condition_label, n_cond, sfreq, out_file, itc_map.shape, edges))

                if smoke:
                    return rows, report_rows
    return rows, report_rows


## Smoke Run

This computes the first available requested-ROI channel for one subject and writes one `.npz` plus one metadata row. Leave `run_smoke=True` for the first pass; set it to `False` after the environment and paths are confirmed.

In [ ]:
# ---- Smoke run ----
smoke_metadata_rows = []
smoke_report_rows = []

if run_smoke:
    smoke_subjects = [str(smoke_subject)] if smoke_subject is not None else subjects
    for subj in smoke_subjects:
        print('Smoke candidate:', subj)
        rows, report = compute_subject_itc(subj, smoke=True)
        smoke_report_rows.extend(report)
        if len(rows) > 0:
            smoke_metadata_rows.extend(rows)
            print('Smoke success:', rows[0])
            break
    if len(smoke_metadata_rows) == 0:
        print('Smoke run did not find a computable subject/channel. Report:')
        display(pd.DataFrame(smoke_report_rows).head(20))
else:
    print('Smoke run skipped.')

smoke_metadata_df = pd.DataFrame(smoke_metadata_rows)
if len(smoke_metadata_df) > 0:
    display(smoke_metadata_df)
    smoke_path = smoke_metadata_df.iloc[0]['itc_file']
    with np.load(smoke_path, allow_pickle=True) as z:
        print('Smoke ITC shape:', z['itc'].shape)
        print('Smoke freqs:', float(z['freqs'].min()), 'to', float(z['freqs'].max()), '| n=', len(z['freqs']))
        print('Smoke times:', float(z['times'].min()), 'to', float(z['times'].max()), '| n=', len(z['times']))

## Batch ITC Computation

Each computed channel contributes one ITC map for `all_trials`, plus one map per available `rpe_cat` category (`neg`, `neutral`, `pos`) with at least `min_trials` trials. Aggregation later pools channel-level maps across subjects within each ROI.

In [ ]:
# ---- Batch run ----
all_metadata_rows = []
all_report_rows = []

if run_batch:
    print('Subject-level joblib parallelization:')
    print('  subjects:', len(subjects), '| subject_n_jobs:', subject_n_jobs, '| backend:', subject_backend)
    print('  Tensorpac n_jobs per channel:', n_jobs)

    def _compute_subject_job(subject):
        rows, report = compute_subject_itc(subject, smoke=False)
        return str(subject), rows, report

    if int(subject_n_jobs) == 1:
        subject_results = []
        for si, subject in enumerate(subjects, start=1):
            print(f'[{si}/{len(subjects)}] {subject}')
            subject_results.append(_compute_subject_job(subject))
    else:
        subject_results = Parallel(n_jobs=int(subject_n_jobs), backend=subject_backend, verbose=10)(
            delayed(_compute_subject_job)(subject) for subject in subjects
        )

    for subject, rows, report in subject_results:
        all_metadata_rows.extend(rows)
        all_report_rows.extend(report)
        print(f'{subject}: computed/cached rows={len(rows)} | report rows={len(report)}')

    metadata_df = pd.DataFrame(all_metadata_rows)
    report_df = pd.DataFrame(all_report_rows)

    if len(metadata_df) > 0:
        metadata_df = metadata_df.sort_values(
            ['participant', 'roi', 'channel', 'condition_type', 'condition_label'],
            kind='mergesort',
        ).reset_index(drop=True)
        metadata_df.to_csv(metadata_csv, index=False)
        print('Saved metadata:', metadata_csv, '| rows:', len(metadata_df))
    else:
        print('No ITC metadata rows generated.')

    if len(report_df) > 0:
        sort_cols = [c for c in ['participant', 'item_type', 'roi', 'channel', 'condition_type', 'condition_label'] if c in report_df.columns]
        if len(sort_cols) > 0:
            report_df = report_df.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)
        report_df.to_csv(missing_csv, index=False)
        print('Saved missing/skip report:', missing_csv, '| rows:', len(report_df))
else:
    print('Batch run skipped. Loading existing metadata if present.')
    metadata_df = pd.read_csv(metadata_csv) if metadata_csv.exists() else pd.DataFrame()
    report_df = pd.read_csv(missing_csv) if missing_csv.exists() else pd.DataFrame()

if len(metadata_df) > 0:
    display(metadata_df.head())
if len(report_df) > 0:
    display(report_df.head())

In [ ]:
rows

## Schema and Shape Checks

In [ ]:
# ---- Schema checks ----
if 'metadata_df' not in locals() or len(metadata_df) == 0:
    if metadata_csv.exists():
        metadata_df = pd.read_csv(metadata_csv)
    else:
        raise FileNotFoundError(f'Missing metadata CSV: {metadata_csv}')

required_metadata_cols = [
    'participant', 'event', 'roi', 'channel', 'condition_type', 'condition_label',
    'n_trials', 'sfreq', 'itc_file'
]
missing_cols = [c for c in required_metadata_cols if c not in metadata_df.columns]
if missing_cols:
    raise ValueError(f'Metadata CSV missing required columns: {missing_cols}')

computed_meta = metadata_df[metadata_df['itc_file'].astype(str).str.len() > 0].copy()
shape_rows = []
for _, row in computed_meta.iterrows():
    path = Path(row['itc_file'])
    if not path.exists():
        shape_rows.append({**row.to_dict(), 'exists': False, 'shape_ok': False, 'detail': 'missing npz'})
        continue
    with np.load(path, allow_pickle=True) as z:
        itc_shape = tuple(z['itc'].shape)
        freqs_shape = tuple(z['freqs'].shape)
        times_shape = tuple(z['times'].shape)
        shape_ok = (len(itc_shape) == 2) and (itc_shape[0] == freqs_shape[0]) and (itc_shape[1] == times_shape[0])
        shape_rows.append({
            'participant': row['participant'],
            'roi': row['roi'],
            'channel': row['channel'],
            'condition_type': row['condition_type'],
            'condition_label': row['condition_label'],
            'itc_file': row['itc_file'],
            'exists': True,
            'itc_shape': str(itc_shape),
            'freqs_shape': str(freqs_shape),
            'times_shape': str(times_shape),
            'shape_ok': bool(shape_ok),
            'detail': '',
        })

shape_check_df = pd.DataFrame(shape_rows)
shape_check_df.to_csv(shape_check_csv, index=False)
print('Saved shape checks:', shape_check_csv)
print('All shape checks passed:', bool(shape_check_df['shape_ok'].all()) if len(shape_check_df) else False)
display(shape_check_df.head())
if len(shape_check_df) and not bool(shape_check_df['shape_ok'].all()):
    display(shape_check_df[~shape_check_df['shape_ok']].head(20))

## Aggregate Plot Helpers

In [ ]:
# ---- Aggregate plotting helpers ----
def load_itc_stack(meta_subset):
    arrays = []
    first_freqs = None
    first_times = None
    used_rows = []
    skipped = []
    for _, row in meta_subset.iterrows():
        path = Path(row['itc_file'])
        if not path.exists():
            skipped.append({**row.to_dict(), 'reason': 'missing_file'})
            continue
        with np.load(path, allow_pickle=True) as z:
            itc = np.asarray(z['itc'], dtype=np.float32)
            freqs = np.asarray(z['freqs'], dtype=np.float32)
            times = np.asarray(z['times'], dtype=np.float32)
        if first_freqs is None:
            first_freqs = freqs
            first_times = times
        same_grid = (itc.shape == (len(first_freqs), len(first_times))) and np.allclose(freqs, first_freqs) and np.allclose(times, first_times)
        if not same_grid:
            skipped.append({**row.to_dict(), 'reason': 'grid_mismatch'})
            continue
        arrays.append(itc)
        used_rows.append(row.to_dict())
    if len(arrays) == 0:
        return None, None, None, pd.DataFrame(used_rows), pd.DataFrame(skipped)
    return np.stack(arrays, axis=0), first_freqs, first_times, pd.DataFrame(used_rows), pd.DataFrame(skipped)


def plot_mean_itc_heatmap(ax, stack, freqs, times, title):
    mean_map = np.nanmean(stack, axis=0)
    sem_map = np.nanstd(stack, axis=0, ddof=1) / np.sqrt(stack.shape[0]) if stack.shape[0] > 1 else np.zeros_like(mean_map)
    im = ax.pcolormesh(times, freqs, mean_map, shading='auto', cmap='viridis', vmin=0.0, vmax=float(np.nanpercentile(mean_map, 99)))
    for cue_t in [0.0, 1.0]:
        ax.axvline(cue_t, color='white', linestyle='--', linewidth=1.2)
    ax.set_ylim(2, 30)
    ax.set_xlabel('Time from cue onset (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{title}\nn channels={stack.shape[0]}, mean SEM={np.nanmean(sem_map):.4f}', fontsize=10)
    return im


def plot_roi_all_trials(metadata_df, roi):
    sub = metadata_df[
        (metadata_df['roi'].astype(str) == roi)
        & (metadata_df['condition_type'].astype(str) == 'all_trials')
        & (metadata_df['condition_label'].astype(str) == 'all')
    ].copy()
    stack, freqs, times, used, skipped = load_itc_stack(sub)
    if stack is None:
        print(f'No plottable all-trials ITC for {roi}')
        return None, used, skipped
    fig, ax = plt.subplots(1, 1, figsize=(8, 4.5))
    im = plot_mean_itc_heatmap(ax, stack, freqs, times, f'{roi} cue_on ITC | all trials')
    fig.colorbar(im, ax=ax, label='ITC')
    fig.tight_layout()
    out_file = figure_root / f'{roi.lower()}_cue_on_itc_all_trials.png'
    fig.savefig(out_file, dpi=300, bbox_inches='tight')
    print('Saved:', out_file)
    plt.show()
    return fig, used, skipped


def plot_roi_rpe_split(metadata_df, roi):
    cats = ['neg', 'neutral', 'pos']
    n_cols = len(cats)
    fig, axes = plt.subplots(1, n_cols, figsize=(5.2 * n_cols, 4.5), sharey=True)
    axes = np.asarray(axes).reshape(-1)
    any_data = False
    used_parts = []
    skipped_parts = []
    last_im = None
    for ax, cat in zip(axes, cats):
        sub = metadata_df[
            (metadata_df['roi'].astype(str) == roi)
            & (metadata_df['condition_type'].astype(str) == 'rpe_cat')
            & (metadata_df['condition_label'].astype(str) == cat)
        ].copy()
        stack, freqs, times, used, skipped = load_itc_stack(sub)
        used_parts.append(used)
        skipped_parts.append(skipped)
        if stack is None:
            ax.text(0.5, 0.5, f'No {cat} data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{roi} {cat}')
            ax.set_axis_off()
            continue
        any_data = True
        last_im = plot_mean_itc_heatmap(ax, stack, freqs, times, f'{roi} cue_on ITC | {cat}')
    if any_data and last_im is not None:
        fig.colorbar(last_im, ax=axes.tolist(), label='ITC', shrink=0.85)
    fig.suptitle(f'{roi} cue_on ITC by RPE category', y=1.02)
    fig.tight_layout()
    out_file = figure_root / f'{roi.lower()}_cue_on_itc_rpe_cat.png'
    fig.savefig(out_file, dpi=300, bbox_inches='tight')
    print('Saved:', out_file)
    plt.show()
    used = pd.concat([x for x in used_parts if len(x) > 0], ignore_index=True) if any(len(x) > 0 for x in used_parts) else pd.DataFrame()
    skipped = pd.concat([x for x in skipped_parts if len(x) > 0], ignore_index=True) if any(len(x) > 0 for x in skipped_parts) else pd.DataFrame()
    return fig, used, skipped


## Sanity Plots

In [ ]:
# ---- Simple sanity plots ----
computed_meta = metadata_df[metadata_df['itc_file'].astype(str).str.len() > 0].copy()

all_trial_meta = computed_meta[(computed_meta['condition_type'] == 'all_trials') & (computed_meta['condition_label'] == 'all')].copy()
channel_counts = (
    all_trial_meta.groupby(['participant', 'roi'])['channel']
    .nunique()
    .reset_index(name='n_channels')
)

roi_colors = {
    'HPC': '#2a9d8f',
    'AMY': '#8e44ad',
    'OFC': '#f4a261',
    'dmPFC': '#457b9d',
    'dlPFC': '#1d3557',
    'ACC': '#e76f51',
}
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
if len(channel_counts) > 0:
    channel_totals = channel_counts.groupby('roi')['n_channels'].sum().reindex(rois).fillna(0)
    channel_totals.plot(kind='bar', ax=ax, color=[roi_colors.get(r, '#666666') for r in channel_totals.index])
ax.set_xlabel('ROI')
ax.set_ylabel('Total channels')
ax.set_title('Computed cue_on ITC channels per ROI')
fig.tight_layout()
out_file = figure_root / 'sanity_channel_counts_per_roi.png'
fig.savefig(out_file, dpi=300, bbox_inches='tight')
print('Saved:', out_file)
plt.show()

# Trial counts by condition/ROI use one row per computed channel-condition.
trial_counts = computed_meta.groupby(['roi', 'condition_type', 'condition_label'])['n_trials'].agg(['count', 'mean', 'min', 'max']).reset_index()
display(trial_counts)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
plot_counts = computed_meta.copy()
plot_counts['roi_condition'] = plot_counts['roi'].astype(str) + ' | ' + plot_counts['condition_type'].astype(str) + ':' + plot_counts['condition_label'].astype(str)
plot_counts.groupby('roi_condition')['n_trials'].mean().sort_index().plot(kind='bar', ax=ax, color='#457b9d')
ax.set_xlabel('ROI / condition')
ax.set_ylabel('Mean trials per channel map')
ax.set_title('Trials contributing to computed ITC maps')
fig.tight_layout()
out_file = figure_root / 'sanity_trials_per_condition.png'
fig.savefig(out_file, dpi=300, bbox_inches='tight')
print('Saved:', out_file)
plt.show()

if 'report_df' not in locals() or report_df is None:
    report_df = pd.read_csv(missing_csv) if missing_csv.exists() else pd.DataFrame()
if len(report_df) > 0:
    print('Missing/skipped report summary:')
    display(report_df.groupby('item_type').size().reset_index(name='n').sort_values('n', ascending=False))
    display(report_df.head(20))
else:
    print('No missing/skipped rows recorded.')

## Aggregate ROI Heatmaps

In [ ]:
# ---- Aggregate heatmaps ----
all_used = []
all_skipped = []

for roi in rois:
    _, used, skipped = plot_roi_all_trials(metadata_df, roi)
    if len(used) > 0:
        all_used.append(used.assign(plot_type='all_trials'))
    if len(skipped) > 0:
        all_skipped.append(skipped.assign(plot_type='all_trials'))

for roi in rois:
    _, used, skipped = plot_roi_rpe_split(metadata_df, roi)
    if len(used) > 0:
        all_used.append(used.assign(plot_type='rpe_cat'))
    if len(skipped) > 0:
        all_skipped.append(skipped.assign(plot_type='rpe_cat'))

aggregate_used_df = pd.concat(all_used, ignore_index=True) if len(all_used) > 0 else pd.DataFrame()
aggregate_skipped_df = pd.concat(all_skipped, ignore_index=True) if len(all_skipped) > 0 else pd.DataFrame()

if len(aggregate_skipped_df) > 0:
    out_file = analysis_root / 'cue_on_itc_aggregate_grid_skips.csv'
    aggregate_skipped_df.to_csv(out_file, index=False)
    print('Saved aggregate grid skips:', out_file)

print('Aggregate maps used:', len(aggregate_used_df))
display(aggregate_used_df.head())

## Notes

- `all_trials` does not depend on behavior columns and will run even when `rpe_cat` is unavailable.
- `rpe_cat` labels are normalized to `neg`, `neutral`, and `pos`.
- Categories with fewer than `min_trials` trials are skipped and recorded in `cue_on_itc_missing_subject_channel_report.csv`.
- Aggregation pools channel-level ITC maps across subjects within each ROI. Metadata preserves participant, channel, condition, trial count, sampling rate, and file path for later subject-level averaging without recomputing ITC.
- Plots use cue markers at 0.0 and 1.0 seconds and the 2-30 Hz phase-frequency range.

## 4-8 Hz ITC Time Courses

Extract channel-level ITC averaged over 4-8 Hz and plot ROI-level time courses across all trials and split by RPE category.

In [ ]:
# ---- 4-8 Hz ITC time courses: all trials and RPE split ----
itc_band = (4.0, 8.0)
timecourse_csv = analysis_root / 'cue_on_itc_4_8hz_timecourses_long.csv'

def extract_band_itc_timecourse(row, band=(4.0, 8.0)):
    path = Path(row['itc_file'])
    if not path.exists():
        return None
    with np.load(path, allow_pickle=True) as z:
        itc = np.asarray(z['itc'], dtype=np.float32)
        freqs = np.asarray(z['freqs'], dtype=np.float32)
        times = np.asarray(z['times'], dtype=np.float32)
    fmask = (freqs >= float(band[0])) & (freqs <= float(band[1]))
    if int(fmask.sum()) == 0:
        return None
    y = np.nanmean(itc[fmask, :], axis=0)
    return pd.DataFrame({
        'participant': str(row['participant']),
        'event': str(row['event']),
        'roi': str(row['roi']),
        'channel': str(row['channel']),
        'condition_type': str(row['condition_type']),
        'condition_label': str(row['condition_label']),
        'n_trials': int(row['n_trials']),
        'sfreq': float(row['sfreq']),
        'time': times.astype(float),
        'itc_4_8hz': y.astype(float),
        'itc_file': str(path),
    })

if 'metadata_df' not in locals() or len(metadata_df) == 0:
    if metadata_csv.exists():
        metadata_df = pd.read_csv(metadata_csv)
    else:
        raise FileNotFoundError(f'Missing metadata CSV: {metadata_csv}')

computed_meta = metadata_df[metadata_df['itc_file'].astype(str).str.len() > 0].copy()
timecourse_parts = []
for _, row in computed_meta.iterrows():
    tc = extract_band_itc_timecourse(row, band=itc_band)
    if tc is not None and len(tc) > 0:
        timecourse_parts.append(tc)

if len(timecourse_parts) == 0:
    raise ValueError('No 4-8 Hz ITC time courses could be extracted from computed metadata.')

itc_timecourse_df = pd.concat(timecourse_parts, ignore_index=True)
itc_timecourse_df.to_csv(timecourse_csv, index=False)
print('Saved 4-8 Hz ITC timecourses:', timecourse_csv, '| rows:', len(itc_timecourse_df))
display(itc_timecourse_df.head())


def summarize_timecourses(df, condition_type, condition_labels):
    sub = df[df['condition_type'].astype(str) == condition_type].copy()
    sub = sub[sub['condition_label'].astype(str).isin(condition_labels)].copy()
    if len(sub) == 0:
        return pd.DataFrame()
    summary = (
        sub.groupby(['roi', 'condition_label', 'time'], dropna=False)['itc_4_8hz']
        .agg(['mean', 'sem', 'count'])
        .reset_index()
        .rename(columns={'count': 'n_channel_maps'})
    )
    summary['sem'] = summary['sem'].fillna(0.0)
    return summary

all_summary = summarize_timecourses(itc_timecourse_df, 'all_trials', ['all'])
rpe_summary = summarize_timecourses(itc_timecourse_df, 'rpe_cat', ['neg', 'neutral', 'pos'])

# All-trials plot: all configured ROIs overlaid.
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
roi_colors = {
    'HPC': '#2a9d8f',
    'AMY': '#8e44ad',
    'OFC': '#f4a261',
    'dmPFC': '#457b9d',
    'dlPFC': '#1d3557',
    'ACC': '#e76f51',
}
for roi in rois:
    roi_df = all_summary[(all_summary['roi'].astype(str) == roi) & (all_summary['condition_label'].astype(str) == 'all')].sort_values('time')
    if len(roi_df) == 0:
        continue
    x = roi_df['time'].to_numpy(dtype=float)
    y = roi_df['mean'].to_numpy(dtype=float)
    sem = roi_df['sem'].to_numpy(dtype=float)
    color = roi_colors.get(roi, None)
    n_maps = int(roi_df['n_channel_maps'].max()) if len(roi_df) else 0
    ax.plot(x, y, linewidth=2, color=color, label=f'{roi} (n={n_maps} maps)')
    ax.fill_between(x, y - sem, y + sem, alpha=0.16, color=color)
for cue_t in [0.0, 1.0]:
    ax.axvline(cue_t, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Time from cue onset (s)')
ax.set_ylabel('Mean ITC, 4-8 Hz')
ax.set_title('Cue-on ITC time course | all trials')
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
out_file = figure_root / 'cue_on_itc_4_8hz_timecourse_all_trials_all_rois.png'
fig.savefig(out_file, dpi=300, bbox_inches='tight')
print('Saved:', out_file)
plt.show()

# RPE split plot: one panel per ROI, conditions overlaid.
rpe_order = ['neg', 'neutral', 'pos']
rpe_colors = {'neg': '#c0392b', 'neutral': '#7f8c8d', 'pos': '#27ae60'}
n_cols = 3
n_rows = int(math.ceil(len(rois) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6.2 * n_cols, 4.2 * n_rows), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(-1)
for ax_idx, ax in enumerate(axes):
    if ax_idx >= len(rois):
        ax.axis('off')
        continue
    roi = rois[ax_idx]
    plotted = False
    for cat in rpe_order:
        cat_df = rpe_summary[(rpe_summary['roi'].astype(str) == roi) & (rpe_summary['condition_label'].astype(str) == cat)].sort_values('time')
        if len(cat_df) == 0:
            continue
        x = cat_df['time'].to_numpy(dtype=float)
        y = cat_df['mean'].to_numpy(dtype=float)
        sem = cat_df['sem'].to_numpy(dtype=float)
        n_maps = int(cat_df['n_channel_maps'].max()) if len(cat_df) else 0
        color = rpe_colors.get(cat, None)
        ax.plot(x, y, linewidth=2, color=color, label=f'{cat} (n={n_maps})')
        ax.fill_between(x, y - sem, y + sem, alpha=0.18, color=color)
        plotted = True
    for cue_t in [0.0, 1.0]:
        ax.axvline(cue_t, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Time from cue onset (s)')
    ax.set_title(f'{roi} | RPE split')
    if plotted:
        ax.legend(frameon=False, fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No RPE-split ITC maps', ha='center', va='center', transform=ax.transAxes)
for ax in axes[::n_cols]:
    ax.set_ylabel('Mean ITC, 4-8 Hz')
fig.suptitle('Cue-on ITC time course by RPE category', y=1.01)
fig.tight_layout()
out_file = figure_root / 'cue_on_itc_4_8hz_timecourse_rpe_cat_all_rois.png'
fig.savefig(out_file, dpi=300, bbox_inches='tight')
print('Saved:', out_file)
plt.show()
